# 02. Data Preparation for Leakage-Safe GNN

This notebook rewrites the old GNN data-preparation notebook and fixes two methodological problems:

1. **Transductive graph leakage**: validation/test exposure edges must not be used in the message-passing graph that creates validation/test node embeddings.
2. **Categorical IDs as continuous numbers**: IDs and category labels must not be placed into a dense continuous feature matrix as if their numerical distance were meaningful.

The output is still a graph dataset for multi-head response prediction, but the feature representation is now split into three tensors:

| Tensor | Shape | Meaning |
|---|---:|---|
| `x_cont` | `[E, D_cont]` | Continuous edge features standardized using train rows only. |
| `x_bin` | `[E, D_bin]` | Binary 0/1 edge features, not standardized. |
| `x_cat` | `[E, D_cat]` | Categorical IDs for embedding lookup, with `0 = UNK`. |
| `edge_index` | `[2, E]` | Directed query edges `user -> item`; used to score candidate edges. |
| `edge_index_train_mp` | `[2, 2E_train]` | Undirected train-only message-passing graph. |
| `edge_index_val_mp` | `[2, 2E_train]` | Undirected validation context graph; train edges only. |
| `edge_index_test_mp` | `[2, ...]` | Undirected test context graph; train-only or train+validation depending on policy. |
| `y` | `[E, 4]` | Observed response heads: complete, long, rewatch, negative. |

Final outputs, if `WRITE_OUTPUTS = True`:

- `KuaiRec 2.0/data/processed/gnn_data.pt`
- `KuaiRec 2.0/data/processed/gnn_data_tiny.pt`

**Important compatibility note:** this fixed dataset is not compatible with the old training notebook that expects a single dense `edge_attr`. The next training notebook should consume `x_cont`, `x_bin`, `x_cat`, and the split-specific message-passing graphs.


## Table of Contents

1. Configure paths, feature lists, and policies.
2. Load processed data.
3. Create chronological train/validation/test edge splits.
4. Build graph query edges and split-specific message-passing graphs.
5. Encode continuous, binary, and categorical features correctly.
6. Validate the artifact.
7. Save full and tiny datasets.


## 0. Setup

Two policies matter most:

| Policy | Default | Explanation |
|---|---|---|
| `TEST_CONTEXT_POLICY` | `"train_val"` | Test node embeddings may use train and validation edges as observed context, but never test edges. Use `"train_only"` for the stricter version. |
| `CATEGORICAL_ENCODING` | `"embedding"` | Category mappings are fitted on training rows only; unseen validation/test values map to `UNK = 0`. For very high-cardinality variables, hashing can be turned on. |


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
BASE = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"

INPUT_PATH = BASE / "processed_data.parquet"
OUTPUT_PATH = BASE / "gnn_data.pt"
TINY_OUTPUT_PATH = BASE / "gnn_data_tiny.pt"

WRITE_OUTPUTS = False  # set True only when you are ready to overwrite the GNN artifacts
RANDOM_SEED = 20260613
TINY_FRAC = 0.05

TEST_CONTEXT_POLICY = "train_val"  # choices: "train_only", "train_val"
CATEGORICAL_ENCODING = "embedding"  # choices: "embedding", "hash_embedding"
MAX_CARDINALITY_WITHOUT_HASHING = 50_000
HASH_BUCKET_SIZE = 10_000

LABEL_COLS = ["y_complete", "y_long", "y_rewatch", "y_neg"]
GRAPH_ID_COLS = ["user_id", "video_id"]
RAW_TIME_COLS = ["time", "date", "timestamp"]
OUTCOME_COLS = ["play_duration", "watch_ratio"] + LABEL_COLS
NOT_SERVING_SAFE_COLS = ["sess_len"]
HUMAN_READABLE_ONLY_COLS = ["i_cat_level1_name", "i_cat_level2_name", "i_cat_level3_name"]

FORBIDDEN_FEATURE_COLS = set(
    GRAPH_ID_COLS
    + RAW_TIME_COLS
    + OUTCOME_COLS
    + NOT_SERVING_SAFE_COLS
    + HUMAN_READABLE_ONLY_COLS
)

if TEST_CONTEXT_POLICY not in {"train_only", "train_val"}:
    raise ValueError("TEST_CONTEXT_POLICY must be 'train_only' or 'train_val'")
if CATEGORICAL_ENCODING not in {"embedding", "hash_embedding"}:
    raise ValueError("CATEGORICAL_ENCODING must be 'embedding' or 'hash_embedding'")

print("input:", INPUT_PATH)
print("write outputs:", WRITE_OUTPUTS)
print("test context policy:", TEST_CONTEXT_POLICY)


## 1. Feature Lists

The old notebook encoded many categorical IDs as scalar numerical features. This notebook separates features by type.

### Continuous features

These have meaningful magnitudes. They are standardized using **training rows only**:

| Feature group | Columns |
|---|---|
| User counts | `u_follow_user_num`, `u_fans_user_num`, `u_friend_user_num`, `u_register_days` |
| User log counts | `u_follow_user_num_log1p`, `u_fans_user_num_log1p`, `u_friend_user_num_log1p`, `u_register_days_log1p` |
| Item continuous | `i_aspect_ratio`, `i_video_duration`, `i_age_since_upload_days` |
| Context continuous | `ctx_hour_sin`, `ctx_hour_cos` |
| Shifted history | `hist_ema_y_complete`, `hist_ema_y_long`, `hist_ema_y_rewatch`, `hist_ema_y_neg`, `hist_ema_watchratio`, `hist_cat_ema_complete`, `hist_cat_entropy_l2`, `hist_author_recency_days`, `hist_prev_sess_len`, `hist_intersess_gap_h` |

### Binary features

These stay as 0/1 floats and are not standardized:

`u_is_lowactive_period`, `u_is_live_streamer`, `u_is_video_author`, `ctx_is_weekend`, `hist_last_complete_author`, `hist_has_author_history`.

### Categorical features

These must be represented by embeddings or hashing, not continuous scalar values:

user activity/range fields, encrypted user tags, item author/music/tag/type IDs, and category IDs.

`session` is also categorical-like: it is an ID, not a continuous amount. `sess_rank` is serving-safe but omitted by default to keep the current feature surface smaller. `sess_len` is excluded because final session length is only known after the session ends.


In [ ]:
CONTINUOUS_COLS = [
    "u_follow_user_num",
    "u_fans_user_num",
    "u_friend_user_num",
    "u_register_days",
    "u_follow_user_num_log1p",
    "u_fans_user_num_log1p",
    "u_friend_user_num_log1p",
    "u_register_days_log1p",
    "i_aspect_ratio",
    "i_video_duration",
    "i_age_since_upload_days",
    "ctx_hour_sin",
    "ctx_hour_cos",
    "hist_ema_y_complete",
    "hist_ema_y_long",
    "hist_ema_y_rewatch",
    "hist_ema_y_neg",
    "hist_ema_watchratio",
    "hist_cat_ema_complete",
    "hist_cat_entropy_l2",
    "hist_author_recency_days",
    "hist_prev_sess_len",
    "hist_intersess_gap_h",
]

BINARY_COLS = [
    "u_is_lowactive_period",
    "u_is_live_streamer",
    "u_is_video_author",
    "ctx_is_weekend",
    "hist_last_complete_author",
    "hist_has_author_history",
]

CATEGORICAL_COLS = [
    "burst_id",
    "session",
    "u_user_active_degree",
    "u_follow_user_num_range",
    "u_fans_user_num_range",
    "u_friend_user_num_range",
    "u_register_days_range",
    *[f"u_onehot_feat{i}" for i in range(18)],
    "i_author_id",
    "i_video_type",
    "i_upload_type",
    "i_visible_status",
    "i_music_id",
    "i_video_tag_id",
    "i_video_tag_name",
    "i_cat_level1_id",
    "i_cat_level2_id",
    "i_cat_level3_id",
]

FEATURE_GROUPS = {
    "continuous": CONTINUOUS_COLS,
    "binary": BINARY_COLS,
    "categorical": CATEGORICAL_COLS,
}

def audit_feature_lists(continuous_cols, binary_cols, categorical_cols, forbidden_cols):
    groups = {
        "continuous": list(continuous_cols),
        "binary": list(binary_cols),
        "categorical": list(categorical_cols),
    }

    overlaps_with_forbidden = {
        group: sorted(set(cols) & set(forbidden_cols))
        for group, cols in groups.items()
        if set(cols) & set(forbidden_cols)
    }
    if overlaps_with_forbidden:
        raise ValueError(f"Forbidden columns appear in feature lists: {overlaps_with_forbidden}")

    repeated = {}
    names = list(groups)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            overlap = sorted(set(groups[left]) & set(groups[right]))
            if overlap:
                repeated[f"{left}/{right}"] = overlap
    if repeated:
        raise ValueError(f"Feature appears in multiple groups: {repeated}")

audit_feature_lists(CONTINUOUS_COLS, BINARY_COLS, CATEGORICAL_COLS, FORBIDDEN_FEATURE_COLS)

feature_table = []
for group, cols in FEATURE_GROUPS.items():
    for col in cols:
        feature_table.append({"group": group, "feature": col})
feature_table = pd.DataFrame(feature_table)
display(feature_table)
print("continuous:", len(CONTINUOUS_COLS), "binary:", len(BINARY_COLS), "categorical:", len(CATEGORICAL_COLS))


## 2. Load Processed Data

We load only the columns needed for graph construction, splits, labels, and feature encoding. This avoids accidentally introducing a new numeric column as a feature.


In [ ]:
required_cols = sorted(set(GRAPH_ID_COLS + RAW_TIME_COLS + LABEL_COLS + CONTINUOUS_COLS + BINARY_COLS + CATEGORICAL_COLS))
df = pd.read_parquet(INPUT_PATH, columns=required_cols)

missing = sorted(set(required_cols) - set(df.columns))
if missing:
    raise KeyError(f"Missing required columns: {missing}")

if df[GRAPH_ID_COLS + ["time"]].isna().any().any():
    raise ValueError("Missing user_id, video_id, or time")

for col in LABEL_COLS:
    values = set(pd.Series(df[col].dropna().unique()).astype(int).tolist())
    if not values.issubset({0, 1}):
        raise ValueError(f"{col} is not binary: {sorted(values)[:10]}")
    if df[col].isna().any():
        raise ValueError(f"{col} contains missing values")

print("rows:", f"{len(df):,}")
print("columns loaded:", len(df.columns))
print("time range:", df["time"].min(), "to", df["time"].max())
display(df.head())


## 3. Global Session-Level Chronological Split

We now define train/validation/test at the **session level**, not by burst days and not by individual interaction rows.

Construction:

1. For each `(user_id, session)`, compute:

   ```text
   session_start_time = min interaction time inside that session
   session_end_time   = max interaction time inside that session
   ```

2. Sort all sessions globally by:

   ```text
   session_start_time, user_id, session
   ```

   The `user_id, session` tie-breaker makes the split deterministic if two sessions start at the same timestamp.

3. Assign:

   | Split | Rule |
   |---|---|
   | Train | first 80% of sessions |
   | Validation | next 10% of sessions |
   | Test | final 10% of sessions |

4. Propagate the session split label back to every interaction row.

This prevents rows from the same session being split across train/validation/test. Message-passing context is then built from session splits:

- validation message passing uses train sessions only;
- test message passing uses train + validation sessions by default;
- validation query edges are never in validation message passing;
- test query edges are never in test message passing.

Important nuance: the chronological boundary is based on `session_start_time`, exactly as defined above. Because sessions have duration and users overlap in real time, row-level timestamps near the boundary can overlap slightly; the hard split unit is the session.


In [ ]:
def make_session_level_chronological_split(frame):
    required = ["user_id", "session", "time"]
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise KeyError(f"Cannot build session split; missing columns: {missing}")

    session_table = (
        frame.reset_index(names="row_id")
        .groupby(["user_id", "session"], observed=True, sort=False)
        .agg(
            session_start_time=("time", "min"),
            session_end_time=("time", "max"),
            n_interactions=("row_id", "size"),
            first_row_id=("row_id", "min"),
        )
        .reset_index()
        .sort_values(["session_start_time", "user_id", "session"], kind="mergesort")
        .reset_index(drop=True)
    )

    n_sessions = len(session_table)
    if n_sessions == 0:
        raise ValueError("No sessions found")

    n_train_sessions = int(np.floor(0.80 * n_sessions))
    n_val_sessions = int(np.floor(0.10 * n_sessions))
    n_test_sessions = n_sessions - n_train_sessions - n_val_sessions

    if min(n_train_sessions, n_val_sessions, n_test_sessions) <= 0:
        raise ValueError(
            "Session split produced an empty split: "
            f"train={n_train_sessions}, val={n_val_sessions}, test={n_test_sessions}"
        )

    session_table["split"] = "test"
    session_table.loc[:n_train_sessions - 1, "split"] = "train"
    session_table.loc[n_train_sessions:n_train_sessions + n_val_sessions - 1, "split"] = "val"

    # Stable session key for mapping split labels back to interaction rows.
    session_table["session_key"] = (
        session_table["user_id"].astype("string")
        + "||"
        + session_table["session"].astype("string")
    )
    session_split_map = session_table.set_index("session_key")["split"]

    row_session_key = (
        frame["user_id"].astype("string")
        + "||"
        + frame["session"].astype("string")
    )
    split_label = row_session_key.map(session_split_map)
    if split_label.isna().any():
        raise ValueError("Some interaction rows did not receive a session split label")

    train_mask = split_label.eq("train").to_numpy()
    val_mask = split_label.eq("val").to_numpy()
    test_mask = split_label.eq("test").to_numpy()

    if np.any(train_mask & val_mask) or np.any(train_mask & test_mask) or np.any(val_mask & test_mask):
        raise ValueError("Split masks overlap")
    if not np.all(train_mask | val_mask | test_mask):
        raise ValueError("Some rows were not assigned to any split")

    # Hard check: each session appears in exactly one split.
    row_split_check = pd.DataFrame({
        "user_id": frame["user_id"].to_numpy(),
        "session": frame["session"].to_numpy(),
        "split": split_label.to_numpy(),
    })
    split_counts_per_session = (
        row_split_check.drop_duplicates()
        .groupby(["user_id", "session"], observed=True)["split"]
        .nunique()
    )
    if split_counts_per_session.gt(1).any():
        raise ValueError("At least one session was assigned to more than one split")

    train_sessions = session_table[session_table["split"].eq("train")]
    val_sessions = session_table[session_table["split"].eq("val")]
    test_sessions = session_table[session_table["split"].eq("test")]

    if not (
        train_sessions["session_start_time"].max()
        <= val_sessions["session_start_time"].min()
        <= val_sessions["session_start_time"].max()
        <= test_sessions["session_start_time"].min()
    ):
        raise ValueError("Session start times are not globally chronological across splits")

    split_summary = (
        session_table.groupby("split", sort=False)
        .agg(
            n_sessions=("session_key", "size"),
            n_interactions=("n_interactions", "sum"),
            session_start_min=("session_start_time", "min"),
            session_start_max=("session_start_time", "max"),
            session_end_max=("session_end_time", "max"),
        )
        .reindex(["train", "val", "test"])
        .reset_index()
    )

    return (
        np.where(train_mask)[0].astype("int64"),
        np.where(val_mask)[0].astype("int64"),
        np.where(test_mask)[0].astype("int64"),
        split_label.astype("string"),
        row_session_key.astype("string"),
        session_table,
        split_summary,
    )


(
    train_idx,
    val_idx,
    test_idx,
    split_label,
    row_session_key,
    session_table,
    split_summary,
) = make_session_level_chronological_split(df)

df = df.copy()
df["gnn_split"] = split_label
df["session_key"] = row_session_key

print("train/val/test interactions:", len(train_idx), len(val_idx), len(test_idx))
print("train/val/test sessions:")
display(split_summary)
print("First and last few sessions after global chronological sorting:")
display(pd.concat([session_table.head(5), session_table.tail(5)], axis=0))


## 4. Graph Edges and Leakage-Safe Session Context Graphs

The query graph stores every observed exposure edge:

```text
edge_index[:, e] = [user_id_e, n_users + video_id_e]
```

Message passing uses session-based context graphs:

| Context graph | Contains | Does not contain |
|---|---|---|
| `edge_index_train_context` | edges from train sessions | validation/test query edges |
| `edge_index_val_context` | edges from train sessions only | validation/test query edges |
| `edge_index_test_context` with `"train_val"` | edges from train + validation sessions | test query edges |
| `edge_index_test_context` with `"train_only"` | edges from train sessions only | validation/test query edges |

Each context graph is made undirected for GCN message passing by concatenating `(u, v)` and `(v, u)`.

The hard leakage checks below are event-index checks. The same user-video pair can appear in multiple sessions or periods; that is reported as a diagnostic, but future **edge events** are still excluded from the relevant context graph.


In [ ]:
n_users = int(df["user_id"].max()) + 1
n_items = int(df["video_id"].max()) + 1
num_nodes = n_users + n_items
num_edges = len(df)

src = df["user_id"].to_numpy(dtype="int64")
dst = n_users + df["video_id"].to_numpy(dtype="int64")
edge_index = torch.from_numpy(np.vstack([src, dst])).long()

def make_undirected(edge_index_directed):
    return torch.cat([edge_index_directed, edge_index_directed.flip(0)], dim=1)

def pair_set(edge_index_directed):
    return set(map(tuple, edge_index_directed.t().tolist()))

def edge_event_leakage_check(context_idx, forbidden_idx, name):
    context_events = set(np.asarray(context_idx, dtype="int64").tolist())
    forbidden_events = set(np.asarray(forbidden_idx, dtype="int64").tolist())
    overlap = context_events & forbidden_events
    if overlap:
        raise ValueError(f"{name} contains {len(overlap)} forbidden edge events")

train_context_idx = train_idx
val_context_idx = train_idx
if TEST_CONTEXT_POLICY == "train_val":
    test_context_idx = np.concatenate([train_idx, val_idx])
else:
    test_context_idx = train_idx

# Required leakage checks.
edge_event_leakage_check(train_context_idx, val_idx, "train_context vs validation query edges")
edge_event_leakage_check(train_context_idx, test_idx, "train_context vs test query edges")
edge_event_leakage_check(val_context_idx, val_idx, "validation context vs validation query edges")
edge_event_leakage_check(val_context_idx, test_idx, "validation context vs test query edges")
edge_event_leakage_check(test_context_idx, test_idx, "test context vs test query edges")

# Session-level context checks.
train_context_splits = set(df.iloc[train_context_idx]["gnn_split"].unique().tolist())
val_context_splits = set(df.iloc[val_context_idx]["gnn_split"].unique().tolist())
test_context_splits = set(df.iloc[test_context_idx]["gnn_split"].unique().tolist())

if train_context_splits != {"train"}:
    raise ValueError(f"Train context has unexpected splits: {train_context_splits}")
if val_context_splits != {"train"}:
    raise ValueError(f"Validation context has unexpected splits: {val_context_splits}")
expected_test_context_splits = {"train", "val"} if TEST_CONTEXT_POLICY == "train_val" else {"train"}
if test_context_splits != expected_test_context_splits:
    raise ValueError(f"Test context has unexpected splits: {test_context_splits}")

edge_index_train_context = edge_index[:, train_context_idx]
edge_index_val_context = edge_index[:, val_context_idx]
edge_index_test_context = edge_index[:, test_context_idx]

edge_index_train_mp = make_undirected(edge_index_train_context)
edge_index_val_mp = make_undirected(edge_index_val_context)
edge_index_test_mp = make_undirected(edge_index_test_context)

# Pair overlap is diagnostic only: repeated user-video pairs across time are real data,
# but future edge events are still excluded from the context graph.
train_pairs = pair_set(edge_index[:, train_idx])
val_pairs = pair_set(edge_index[:, val_idx])
test_pairs = pair_set(edge_index[:, test_idx])
pair_overlap_report = pd.DataFrame([
    {"comparison": "train query pairs vs validation query pairs", "n_pair_overlap": len(train_pairs & val_pairs)},
    {"comparison": "train query pairs vs test query pairs", "n_pair_overlap": len(train_pairs & test_pairs)},
    {"comparison": "train+validation query pairs vs test query pairs", "n_pair_overlap": len((train_pairs | val_pairs) & test_pairs)},
])

def assert_undirected(edge_index_mp, name):
    pairs = pair_set(edge_index_mp)
    missing_reverse = [(u, v) for (u, v) in list(pairs)[:10000] if (v, u) not in pairs]
    if missing_reverse:
        raise ValueError(f"{name} is not undirected; first missing reverse: {missing_reverse[0]}")

assert_undirected(edge_index_train_mp, "edge_index_train_mp")
assert_undirected(edge_index_val_mp, "edge_index_val_mp")
assert_undirected(edge_index_test_mp, "edge_index_test_mp")

print("n_users:", n_users, "n_items:", n_items, "num_nodes:", num_nodes)
print("query edge_index:", tuple(edge_index.shape))
print("message graph edges:", {
    "train_mp": edge_index_train_mp.shape[1],
    "val_mp": edge_index_val_mp.shape[1],
    "test_mp": edge_index_test_mp.shape[1],
})
print("context split sets:", {
    "train_context": sorted(train_context_splits),
    "val_context": sorted(val_context_splits),
    "test_context": sorted(test_context_splits),
})
display(pair_overlap_report)


## 5. Feature Encoding

### Continuous features

For each continuous feature `x`, fit mean and standard deviation on training rows only:

```text
z_e = (x_e - mean_train(x)) / (std_train(x) + 1e-8)
```

Missing continuous values are filled with the training mean before scaling, so they become zero after scaling.

### Binary features

Binary features are clipped to `[0, 1]`, missing values are filled with 0, and no standardization is applied.

### Categorical features

Category mappings are fitted on training rows only:

```text
UNK = 0
known train categories = 1, 2, ...
unseen validation/test categories -> 0
```

The training notebook should pass `x_cat` into embedding layers, not concatenate it as continuous scalars.


In [ ]:
def fit_continuous(frame, train_idx, cols):
    train = frame.iloc[train_idx]
    x = pd.DataFrame(index=frame.index)
    scaler = {}
    missing = {}
    for col in cols:
        raw_all = pd.to_numeric(frame[col], errors="coerce").astype("float32")
        raw_train = pd.to_numeric(train[col], errors="coerce").astype("float32")
        mean = float(raw_train.mean(skipna=True))
        std = float(raw_train.std(skipna=True))
        if not np.isfinite(mean):
            mean = 0.0
        if not np.isfinite(std) or std <= 1e-8:
            std = 1.0
        x[col] = ((raw_all.fillna(mean) - mean) / (std + 1e-8)).astype("float32")
        scaler[col] = {"mean": mean, "std": std}
        missing[col] = int(raw_all.isna().sum())
    return torch.from_numpy(x[cols].to_numpy(dtype="float32")), scaler, missing

def fit_binary(frame, cols):
    x = pd.DataFrame(index=frame.index)
    missing = {}
    for col in cols:
        raw = pd.to_numeric(frame[col], errors="coerce")
        x[col] = raw.fillna(0).clip(0, 1).astype("float32")
        missing[col] = int(raw.isna().sum())
    return torch.from_numpy(x[cols].to_numpy(dtype="float32")), missing

def stable_hash_codes(values, n_buckets):
    hashed = pd.util.hash_pandas_object(values.astype("string").fillna("__MISSING__"), index=False)
    return (hashed.to_numpy(dtype="uint64") % np.uint64(n_buckets)).astype("int64") + 1

def fit_categorical(frame, train_idx, cols):
    train = frame.iloc[train_idx]
    arrays = []
    mappings = {}
    cardinalities = []

    for col in cols:
        train_values = train[col].astype("string").fillna("__MISSING__").replace({"": "__MISSING__"})
        all_values = frame[col].astype("string").fillna("__MISSING__").replace({"": "__MISSING__"})
        n_train_levels = int(train_values.nunique(dropna=False))

        use_hash = (
            CATEGORICAL_ENCODING == "hash_embedding"
            or n_train_levels > MAX_CARDINALITY_WITHOUT_HASHING
        )

        if use_hash:
            codes = stable_hash_codes(all_values, HASH_BUCKET_SIZE)
            mapping = None
            cardinality = HASH_BUCKET_SIZE + 1
            encoding = "hash_embedding"
        else:
            levels = sorted(train_values.unique().tolist())
            mapping = {value: i + 1 for i, value in enumerate(levels)}
            codes = all_values.map(mapping).fillna(0).astype("int64").to_numpy()
            cardinality = len(levels) + 1
            encoding = "embedding"

        arrays.append(codes.reshape(-1, 1))
        cardinalities.append(cardinality)
        mappings[col] = {
            "encoding": encoding,
            "cardinality": int(cardinality),
            "n_train_levels": int(n_train_levels),
            "n_unseen_mapped_to_unk": int((codes == 0).sum()),
            "mapping": mapping,
        }

    return torch.from_numpy(np.hstack(arrays).astype("int64")).long(), mappings, cardinalities

x_cont, continuous_scaler, continuous_missing = fit_continuous(df, train_idx, CONTINUOUS_COLS)
x_bin, binary_missing = fit_binary(df, BINARY_COLS)
x_cat, categorical_mappings, categorical_cardinalities = fit_categorical(df, train_idx, CATEGORICAL_COLS)

if not torch.isfinite(x_cont).all():
    raise ValueError("x_cont contains non-finite values")
if not torch.isfinite(x_bin).all():
    raise ValueError("x_bin contains non-finite values")
if x_cat.min().item() < 0:
    raise ValueError("x_cat contains negative category codes")

print("x_cont:", tuple(x_cont.shape))
print("x_bin:", tuple(x_bin.shape))
print("x_cat:", tuple(x_cat.shape))
display(pd.Series(categorical_cardinalities, index=CATEGORICAL_COLS, name="cardinality").sort_values(ascending=False).to_frame())
display(pd.Series(continuous_missing, name="n_missing_continuous").sort_values(ascending=False).to_frame().query("n_missing_continuous > 0"))


## 6. Labels, Package, and Validation

The saved package includes:

- query edges for scoring (`edge_index`);
- split-specific message-passing graphs (`edge_index_*_mp`);
- separate feature tensors (`x_cont`, `x_bin`, `x_cat`);
- train/validation/test edge indices from the global session-level chronological split;
- session split metadata for reproducibility;
- scalers and category mappings.


In [ ]:
y = torch.from_numpy(df[LABEL_COLS].to_numpy(dtype="float32")).float()
train_idx_t = torch.from_numpy(train_idx).long()
val_idx_t = torch.from_numpy(val_idx).long()
test_idx_t = torch.from_numpy(test_idx).long()

package_checks = {
    "edge_index_shape": tuple(edge_index.shape) == (2, num_edges),
    "x_cont_shape": x_cont.shape[0] == num_edges and x_cont.shape[1] == len(CONTINUOUS_COLS),
    "x_bin_shape": x_bin.shape[0] == num_edges and x_bin.shape[1] == len(BINARY_COLS),
    "x_cat_shape": x_cat.shape[0] == num_edges and x_cat.shape[1] == len(CATEGORICAL_COLS),
    "y_shape": tuple(y.shape) == (num_edges, 4),
    "splits_cover_edges": train_idx_t.numel() + val_idx_t.numel() + test_idx_t.numel() == num_edges,
    "val_context_excludes_val_events": len(set(val_context_idx.tolist()) & set(val_idx.tolist())) == 0,
    "test_context_excludes_test_events": len(set(test_context_idx.tolist()) & set(test_idx.tolist())) == 0,
    "each_session_has_one_split": not session_table["session_key"].duplicated().any(),
}
for name, ok in package_checks.items():
    print(f"{name}: {ok}")
failed = [name for name, ok in package_checks.items() if not ok]
if failed:
    raise ValueError(f"Package validation failed: {failed}")

def dataframe_records_for_torch_save(frame):
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].astype("string")
    return out.to_dict(orient="records")


feature_metadata = {
    "continuous_cols": CONTINUOUS_COLS,
    "binary_cols": BINARY_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "continuous_scaler": continuous_scaler,
    "continuous_missing": continuous_missing,
    "binary_missing": binary_missing,
    "categorical_mappings": categorical_mappings,
    "categorical_cardinalities": categorical_cardinalities,
    "categorical_encoding": CATEGORICAL_ENCODING,
    "max_cardinality_without_hashing": MAX_CARDINALITY_WITHOUT_HASHING,
    "hash_bucket_size": HASH_BUCKET_SIZE,
    "forbidden_feature_cols": sorted(FORBIDDEN_FEATURE_COLS),
}

gnn_data = {
    "edge_index": edge_index,
    "edge_index_train_context": edge_index_train_context,
    "edge_index_val_context": edge_index_val_context,
    "edge_index_test_context": edge_index_test_context,
    "edge_index_train_mp": edge_index_train_mp,
    "edge_index_val_mp": edge_index_val_mp,
    "edge_index_test_mp": edge_index_test_mp,
    "x_cont": x_cont,
    "x_bin": x_bin,
    "x_cat": x_cat,
    "y": y,
    "train_idx": train_idx_t,
    "val_idx": val_idx_t,
    "test_idx": test_idx_t,
    "n_users": int(n_users),
    "n_items": int(n_items),
    "num_nodes": int(num_nodes),
    "label_names": LABEL_COLS,
    "feature_metadata": feature_metadata,
    "split_summary": dataframe_records_for_torch_save(split_summary),
    "session_table": dataframe_records_for_torch_save(session_table),
    "test_context_policy": TEST_CONTEXT_POLICY,
}

print("Ready to save clean GNN package.")


## 7. Tiny Dataset and Save

The tiny sample is also session-level. It samples whole sessions separately from train, validation, and test, then includes all interactions from the selected sessions. This preserves the rule that every interaction row in a session belongs to the same split.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

def sample_session_keys(session_keys, frac):
    session_keys = np.asarray(session_keys, dtype=object)
    k = max(1, int(round(frac * len(session_keys))))
    return set(rng.choice(session_keys, size=k, replace=False).tolist())

train_session_keys = session_table.loc[session_table["split"].eq("train"), "session_key"].to_numpy()
val_session_keys = session_table.loc[session_table["split"].eq("val"), "session_key"].to_numpy()
test_session_keys = session_table.loc[session_table["split"].eq("test"), "session_key"].to_numpy()

tiny_train_sessions = sample_session_keys(train_session_keys, TINY_FRAC)
tiny_val_sessions = sample_session_keys(val_session_keys, TINY_FRAC)
tiny_test_sessions = sample_session_keys(test_session_keys, TINY_FRAC)

tiny_train_global = np.where(df["session_key"].isin(tiny_train_sessions).to_numpy())[0].astype("int64")
tiny_val_global = np.where(df["session_key"].isin(tiny_val_sessions).to_numpy())[0].astype("int64")
tiny_test_global = np.where(df["session_key"].isin(tiny_test_sessions).to_numpy())[0].astype("int64")

tiny_global_idx = np.concatenate([tiny_train_global, tiny_val_global, tiny_test_global])
tiny_global_idx_t = torch.from_numpy(tiny_global_idx).long()

n_tiny_train = len(tiny_train_global)
n_tiny_val = len(tiny_val_global)
n_tiny_test = len(tiny_test_global)

tiny_train_idx = torch.arange(0, n_tiny_train, dtype=torch.long)
tiny_val_idx = torch.arange(n_tiny_train, n_tiny_train + n_tiny_val, dtype=torch.long)
tiny_test_idx = torch.arange(n_tiny_train + n_tiny_val, n_tiny_train + n_tiny_val + n_tiny_test, dtype=torch.long)

tiny_edge_index = edge_index[:, tiny_global_idx_t]
tiny_train_context_idx = tiny_train_idx.numpy()
tiny_val_context_idx = tiny_train_idx.numpy()
if TEST_CONTEXT_POLICY == "train_val":
    tiny_test_context_idx = np.concatenate([tiny_train_idx.numpy(), tiny_val_idx.numpy()])
else:
    tiny_test_context_idx = tiny_train_idx.numpy()

if set(tiny_val_context_idx.tolist()) & set(tiny_val_idx.numpy().tolist()):
    raise ValueError("Tiny validation context includes validation query events")
if set(tiny_test_context_idx.tolist()) & set(tiny_test_idx.numpy().tolist()):
    raise ValueError("Tiny test context includes test query events")

tiny_data = {
    "edge_index": tiny_edge_index,
    "edge_index_train_context": tiny_edge_index[:, tiny_train_context_idx],
    "edge_index_val_context": tiny_edge_index[:, tiny_val_context_idx],
    "edge_index_test_context": tiny_edge_index[:, tiny_test_context_idx],
    "edge_index_train_mp": make_undirected(tiny_edge_index[:, tiny_train_context_idx]),
    "edge_index_val_mp": make_undirected(tiny_edge_index[:, tiny_val_context_idx]),
    "edge_index_test_mp": make_undirected(tiny_edge_index[:, tiny_test_context_idx]),
    "x_cont": x_cont[tiny_global_idx_t],
    "x_bin": x_bin[tiny_global_idx_t],
    "x_cat": x_cat[tiny_global_idx_t],
    "y": y[tiny_global_idx_t],
    "train_idx": tiny_train_idx,
    "val_idx": tiny_val_idx,
    "test_idx": tiny_test_idx,
    "n_users": int(n_users),
    "n_items": int(n_items),
    "num_nodes": int(num_nodes),
    "label_names": LABEL_COLS,
    "feature_metadata": feature_metadata,
    "source_full_edge_indices": tiny_global_idx_t,
    "test_context_policy": TEST_CONTEXT_POLICY,
    "tiny_frac": TINY_FRAC,
    "random_seed": RANDOM_SEED,
    "tiny_session_counts": {
        "train": len(tiny_train_sessions),
        "val": len(tiny_val_sessions),
        "test": len(tiny_test_sessions),
    },
}

print("tiny train/val/test interactions:", n_tiny_train, n_tiny_val, n_tiny_test)
print("tiny train/val/test sessions:", len(tiny_train_sessions), len(tiny_val_sessions), len(tiny_test_sessions))
print("tiny total edges:", tiny_edge_index.shape[1])

if WRITE_OUTPUTS:
    torch.save(gnn_data, OUTPUT_PATH)
    torch.save(tiny_data, TINY_OUTPUT_PATH)
    print("saved:", OUTPUT_PATH)
    print("saved:", TINY_OUTPUT_PATH)
else:
    print("WRITE_OUTPUTS is False; skipped saving.")


## Notes for Notebook 3

The training notebook must use the split-specific message-passing graphs:

| Split being scored | Message-passing graph | Query edges |
|---|---|---|
| Train | `edge_index_train_mp` | `edge_index[:, train_idx]` |
| Validation | `edge_index_val_mp` | `edge_index[:, val_idx]` |
| Test | `edge_index_test_mp` | `edge_index[:, test_idx]` |

With the default `TEST_CONTEXT_POLICY = "train_val"`, test message passing uses train + validation sessions but never test sessions. Validation message passing always uses train sessions only.

Categorical features in `x_cat` should go through embedding layers. Do not concatenate `x_cat` to continuous features as raw scalar numbers.
